# FMCG Supply Chain Analytics

## Warehouse-Level Product Weight Prediction

This project explores warehouse-level data from an FMCG instant noodles supply network. I use exploratory analysis and machine learning to understand which warehouse and operational characteristics are associated with the amount of product handled by a warehouse.

**Target variable:** `product_wg_ton`


## 1. Business Problem

FMCG warehouses differ in capacity, location, distribution reach, infrastructure and day-to-day operating conditions. These differences can affect how much product a warehouse handles.

The main question I wanted to answer was:

> **Can the information available about a warehouse be used to predict the product weight handled by that warehouse?**

I also wanted to check whether any feature had an unusually strong relationship with the target before interpreting model performance.


In [ ]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Keep the dataset in the repository's data/ folder.
DATA_PATH = Path('../data/FMCG_data.csv')
df = pd.read_csv(DATA_PATH)

df.head()


## 2. Understanding the Dataset

I first checked the size, data types and structure of the dataset before starting the analysis.


In [ ]:
# Dataset dimensions and data types
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
df.info()


In [ ]:
# Check where values are missing
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]


### Missing Values

There are missing values in `workers_num`, `wh_est_year`, and `approved_wh_govt_certificate`.

For modeling, I handle missing numerical values with median imputation and missing categorical values with the most frequent category. I exclude `wh_est_year` because it has substantial missingness and is not essential to the main analysis.


## 3. Exploratory Data Analysis

I used a few simple plots to understand the target distribution, warehouse coverage across zones, and relationships between numerical variables.


In [ ]:
# Distribution of the target variable
plt.figure(figsize=(8, 5))
plt.hist(df['product_wg_ton'], bins=40)
plt.title('Distribution of Product Weight')
plt.xlabel('Product Weight (ton)')
plt.ylabel('Warehouses')
plt.show()


In [ ]:
# Number of warehouses in each zone
zone_counts = df['zone'].value_counts()

plt.figure(figsize=(8, 5))
plt.bar(zone_counts.index, zone_counts.values)
plt.title('Warehouse Distribution by Zone')
plt.xlabel('Zone')
plt.ylabel('Warehouses')
plt.show()


In [ ]:
# Correlation between numerical variables
numeric_corr = df.select_dtypes(include=np.number).corr()

plt.figure(figsize=(11, 8))
plt.imshow(numeric_corr, cmap='viridis', aspect='auto', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.xticks(range(len(numeric_corr)), numeric_corr.columns, rotation=90, fontsize=7)
plt.yticks(range(len(numeric_corr)), numeric_corr.columns, fontsize=7)
plt.title('Numerical Feature Correlation Heatmap')
plt.tight_layout()
plt.show()


## 4. Checking an Unusually Strong Relationship

During the exploratory analysis, `storage_issue_reported_l3m` showed an unusually strong correlation with `product_wg_ton` (approximately 0.987).

That stood out because operational variables usually would not be expected to explain almost all variation in a continuous target through a simple linear relationship.

Before relying on this feature, I checked its relationship with the target more closely and compared model performance with and without it.


In [ ]:
# Quantify the relationship that stood out during EDA
corr_storage = df['storage_issue_reported_l3m'].corr(df['product_wg_ton'])
print(f'Correlation: {corr_storage:.6f}')

df.groupby('storage_issue_reported_l3m')['product_wg_ton'].agg(
    ['count', 'mean', 'median', 'min', 'max']
).head(10)


In [ ]:
# Plot a sample so the relationship is easier to inspect visually
sample = df.sample(min(5000, len(df)), random_state=42)

plt.figure(figsize=(9, 5))
plt.scatter(
    sample['storage_issue_reported_l3m'],
    sample['product_wg_ton'],
    s=8,
    alpha=0.25
)
plt.title('Storage Issues vs Product Weight')
plt.xlabel('Storage Issues Reported (last 3 months)')
plt.ylabel('Product Weight (ton)')
plt.show()


### What the Check Shows

The relationship is extremely strong in this dataset. Correlation alone does not tell us that storage issues cause higher product weight.

It also raises a practical modeling question: **would this variable be available independently at the exact time a prediction is needed, or could it be closely tied to the process that generated the target?**

To understand the effect, I report two versions of the Random Forest model:

1. A model using the available features, including `storage_issue_reported_l3m`.
2. A second model with that feature removed.


## 5. Preparing the Data for Modeling

I removed the target and identifier columns, separated numerical and categorical variables, and built the preprocessing directly into a scikit-learn pipeline.

This keeps imputation and encoding inside the training workflow and avoids applying transformations to the test data before model fitting.


In [ ]:
# Separate target, identifiers and predictors
TARGET = 'product_wg_ton'
DROP_COLS = ['product_wg_ton', 'Ware_house_ID', 'WH_Manager_ID', 'wh_est_year']

X = df.drop(columns=DROP_COLS)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(exclude='object').columns.tolist()

# Preprocessing is kept inside the pipeline so the test set is not used
# when fitting imputers or encoders.
preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

def evaluate(model, X_train, X_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    return {
        'MAE': mean_absolute_error(y_test, pred),
        'MSE': mean_squared_error(y_test, pred),
        'R2': r2_score(y_test, pred)
    }, pred


## 6. Linear Regression Baseline

I started with Linear Regression as a simple baseline. This gives me a reference point before using a nonlinear model.


In [ ]:
# Linear Regression baseline
linear = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

linear_metrics, linear_pred = evaluate(linear, X_train, X_test)
linear_metrics


## 7. Random Forest Regression

Next, I used a Random Forest Regressor. Random Forest can capture nonlinear relationships and interactions between warehouse characteristics, so it is useful for comparison with the linear baseline.


In [ ]:
# Random Forest model
rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf_metrics, rf_pred = evaluate(rf, X_train, X_test)
rf_metrics


## 8. Random Forest Without the Storage-Issue Feature

Because of the unusually strong relationship identified during EDA, I repeated the Random Forest experiment after removing `storage_issue_reported_l3m`.

This is a diagnostic experiment rather than a claim that the feature is definitely invalid. The purpose is to measure how dependent the model's predictive performance is on that single variable.


In [ ]:
# Repeat the Random Forest experiment without the suspicious feature
X_no_storage = X.drop(columns=['storage_issue_reported_l3m'])

Xns_train, Xns_test, yns_train, yns_test = train_test_split(
    X_no_storage,
    y,
    test_size=0.20,
    random_state=42
)

cat_ns = X_no_storage.select_dtypes(include='object').columns.tolist()
num_ns = X_no_storage.select_dtypes(exclude='object').columns.tolist()

pre_ns = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_ns),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_ns)
])

rf_ns = Pipeline([
    ('preprocessor', pre_ns),
    ('model', RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf_ns.fit(Xns_train, yns_train)
rf_ns_pred = rf_ns.predict(Xns_test)

rf_ns_metrics = {
    'MAE': mean_absolute_error(yns_test, rf_ns_pred),
    'MSE': mean_squared_error(yns_test, rf_ns_pred),
    'R2': r2_score(yns_test, rf_ns_pred)
}

rf_ns_metrics


## 9. Model Comparison


In [ ]:
# Compare all three experiments
comparison = pd.DataFrame([
    {'Model': 'Linear Regression', **linear_metrics},
    {'Model': 'Random Forest — storage included', **rf_metrics},
    {'Model': 'Random Forest — storage excluded', **rf_ns_metrics}
])

comparison


### Results

The storage-inclusive Random Forest performs substantially better on the test set than the same model without `storage_issue_reported_l3m`.

This tells me that the very high predictive performance is strongly dependent on this feature. I would therefore avoid presenting the 0.99+ R² as evidence that the model is ready for real-world deployment.

The more useful takeaway is that the dataset contains a very strong relationship that needs business validation before it can be used for operational decision-making.


## 10. Actual vs Predicted Values

The plots below compare actual product weight with the values predicted by the two Random Forest models.

A point close to the diagonal line represents a prediction close to the actual value.


In [ ]:
# Random Forest predictions with the storage feature included
plt.figure(figsize=(8, 6))
plt.scatter(y_test, rf_pred, s=8, alpha=0.25)

lims = [
    min(y_test.min(), rf_pred.min()),
    max(y_test.max(), rf_pred.max())
]
plt.plot(lims, lims, linestyle='--')

plt.xlabel('Actual Product Weight')
plt.ylabel('Predicted Product Weight')
plt.title('Random Forest — Storage Feature Included')
plt.show()


In [ ]:
# Random Forest predictions after removing the storage-issue feature
plt.figure(figsize=(8, 6))
plt.scatter(yns_test, rf_ns_pred, s=8, alpha=0.25)

lims = [
    min(yns_test.min(), rf_ns_pred.min()),
    max(yns_test.max(), rf_ns_pred.max())
]
plt.plot(lims, lims, linestyle='--')

plt.xlabel('Actual Product Weight')
plt.ylabel('Predicted Product Weight')
plt.title('Random Forest — Storage Feature Excluded')
plt.show()


## 11. Conclusion

The analysis shows that warehouse-level variables contain substantial information about `product_wg_ton`.

The Random Forest model using the full feature set achieved very high test-set performance. However, most of that performance is closely tied to `storage_issue_reported_l3m`, which has an unusually strong relationship with the target.

Removing that feature causes a large drop in performance. I would therefore treat the high-R² model as an exploratory result rather than a production-ready forecasting model.

### Key takeaways

- The dataset contains meaningful operational and warehouse-level information.
- Random Forest captures the target relationship better than the linear baseline in the tested setup.
- `storage_issue_reported_l3m` deserves further business investigation because of its unusually strong association with product weight.
- Before deployment, I would validate feature availability, investigate how the target and storage-issue variable are generated, and test the model on a genuinely independent dataset.

### Possible next steps

- Add time-based or demand-related variables if available.
- Investigate the business meaning and data-generation process of the storage-issue variable.
- Test additional models and hyperparameters.
- Use explainability methods such as permutation importance or SHAP after the feature definitions are validated.
- Extend the prediction model into an inventory or warehouse planning optimization problem.
